## LORA Finetuning - Training

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer
from datasets import Dataset
from peft import PeftModel, get_peft_model, LoraConfig
import pandas as pd

df = pd.read_csv("training_data/lora_training_data.csv")

my_training_input = []
for i in range(df.shape[0]):
    ques_answer_combination = f"""\n Question: {df['question'][i]} \n\n Answer: {df['response'][i]}"""
    my_training_input.append(ques_answer_combination)

training_huggingface_dataset =Dataset.from_dict({"text": my_training_input}) 

lora_config = LoraConfig(
      r = 8,
      lora_alpha=12,
      target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
      bias='none',
      task_type="CAUSAL_LM"
   )

training_args = SFTConfig(
    output_dir="lora_directory",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=3,
    num_train_epochs=3,
    gradient_checkpointing=True,
    disable_tqdm=False,
    learning_rate=0.002,
    fp16=True
)

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype= torch.float16)
peft_model = get_peft_model(base_model, lora_config)
trainer = SFTTrainer(
    model=peft_model,
    train_dataset=training_huggingface_dataset,
    processing_class=tokenizer,
    args=training_args
)

trainer.train()
peft_model.save_pretrained("./lora_peft_model")
tokenizer.save_pretrained("./lora_peft_model")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

## Model Loading & Inferencing

In [ ]:
base_model_loaded = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype = torch.float16)
peft_model_loaded = PeftModel.from_pretrained(base_model_loaded, "./lora_peft_model")

prompt = """Question: Analyze this contract clause: The customer grants the provider a perpetual, irrevocable license to use all customer data for any purpose.

Answer: """

input = tokenizer(prompt, return_tensors="pt").to(peft_model_loaded.device)
with torch.no_grad():
    output = peft_model_loaded.generate(
        **input,
        max_new_tokens = 100,
        temperature = 0.7,
        do_sample = True
    )

result = tokenizer.decode(output[0], skip_special_tokens=True)
print(result)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]